In [1]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader, Subset
from dataset import IMDBDataset, get_celeb_name
from model import AgeClassifier
from torch.utils.data import random_split, SubsetRandomSampler
from torch import optim, nn
from tqdm import tqdm
from unlearn import *
from utils import *
from sklearn.model_selection import train_test_split
import random

In [2]:
teacher_checkpoint_path = 'IMDB_CROP_Pretrained_50000_Samples_Teacher.pt'
forget_checkpoint_path = 'IMDB_CROP_Pretrained_50000_Samples_Forget.pt'

### Unzip Dataset

In [3]:
import tarfile
import os

if not os.path.isdir('./imdb_crop'):
    with tarfile.open('./imdb_crop.tar', 'r') as tar:
        tar.extractall('./')

In [4]:
dataset = IMDBDataset('./imdb.csv', './imdb_crop', 50000) # remove this for full dataset

train_idx, valid_idx = train_test_split(list(range(len(dataset))), test_size=0.2, random_state=42)

train_ds = Subset(dataset, train_idx)
valid_ds = Subset(dataset, valid_idx)

Balanced dataset: 45645 samples across 5 age classes
Unique celebrities: 1324
Class distribution: {0: 9129, 1: 9129, 2: 9129, 3: 9129, 4: 9129}


In [5]:
celeb_ids = torch.tensor(dataset.celeb_ids)
ages = torch.tensor(dataset.ages)

unique_celebs = torch.unique(celeb_ids)
valid_celebs = []

for celeb_id in unique_celebs:
    if torch.any(ages[celeb_ids == celeb_id] <= 13):
        valid_celebs.append(celeb_id.item())

num_celeb = 5
if len(valid_celebs) < num_celeb:
    raise ValueError(f"Only {len(valid_celebs)} celebrities have images where age <= 13, but {num_celeb} requested")

forget_celeb_ids = random.sample(valid_celebs[:10], num_celeb)

In [6]:
print('forget celeb names:')
print(*[get_celeb_name(celeb_id) for celeb_id in forget_celeb_ids], sep='\n')

forget celeb names:
['Abigail Hargrove']
['Adam Harrington']
['Adam Herz']
['Aaron Christian Howles']
['Adal Ramones']


In [7]:
retain_ds = Subset(dataset, torch.tensor([i for i in train_idx if celeb_ids[i].item() not in forget_celeb_ids]))
forget_ds = Subset(dataset, torch.tensor([i for i in train_idx if celeb_ids[i].item() in forget_celeb_ids]))

In [8]:
print('dataset sizes:')
print(len(retain_ds), len(forget_ds), sep='\n')

dataset sizes:
36395
121


In [9]:
device = 'cuda'

batch_size = 256
num_workers = 4

train_dl = DataLoader(train_ds, batch_size, num_workers=num_workers, pin_memory=False, shuffle=True)
valid_dl = DataLoader(valid_ds, batch_size, num_workers=num_workers, pin_memory=False)

retain_dl = DataLoader(retain_ds, batch_size, num_workers=num_workers, pin_memory=False, shuffle = True)
forget_dl = DataLoader(forget_ds, batch_size, num_workers=num_workers, pin_memory=False, shuffle = True)

In [ ]:

full_trained_teacher = AgeClassifier(num_classes = 5, pretrained = True).to(device)

# Training
history = fit_one_cycle(25, full_trained_teacher, train_dl, valid_dl, device = device)

# Loading
# full_trained_teacher.load_state_dict(torch.load(teacher_checkpoint_path, map_location = device))

# Saving
torch.save(full_trained_teacher.state_dict(), teacher_checkpoint_path)

C:\Users\Foopy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\Foopy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
C:\Users\Foopy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\torch\optim\lr_scheduler.py:62: UserWarn

In [ ]:
print(evaluate(full_trained_teacher, train_dl, device)) # overfitted performance
print(evaluate(full_trained_teacher, valid_dl, device)) # unseen performance
print(evaluate(full_trained_teacher, retain_dl, device)) # overfitted performance
print(evaluate(full_trained_teacher, forget_dl, device)) # overfitted performance

### Forget

In [ ]:
model = AgeClassifier(num_classes = 5, pretrained = False).to(device)
unlearning_teacher = AgeClassifier(num_classes = 5, pretrained = False).to(device)

# Training
model.load_state_dict(torch.load(teacher_checkpoint_path, map_location = device))
blindspot_unlearner(model = model, unlearning_teacher = unlearning_teacher, full_trained_teacher = full_trained_teacher, 
                    retain_data = retain_ds, forget_data = forget_ds, epochs = 2, lr = 0.0001, 
                    batch_size = batch_size, num_workers = num_workers, device = device)

# Loading
# model.load_state_dict(torch.load(forget_checkpoint_path, map_location = device))

# Saving
torch.save(model.state_dict(), forget_checkpoint_path)

Epoch 1 Unlearning Loss 0.0273201372474432
Epoch 2 Unlearning Loss 0.024661175906658173


In [ ]:
print(evaluate(model, train_dl, device)) # overfitted performance
print(evaluate(model, valid_dl, device)) # unseen performance
print(evaluate(model, retain_dl, device)) # overfitted performance
print(evaluate(model, forget_dl, device))  # unseen performance